In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/amjadzhour/car-price-prediction/Car_Price_Prediction.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [3]:
dc= pd.read_csv("/kaggle/input/datasets/amjadzhour/car-price-prediction/Car_Price_Prediction.csv")
df= dc.copy()

In [4]:
df["Price"]= df["Price"].round(2)
df

,Make,Model,Year,Engine Size,Mileage,Fuel Type,Transmission,Price
0,Honda,Model B,2015,3.9,74176,Petrol,Manual,30246.21
1,Ford,Model C,2014,1.7,94799,Electric,Automatic,22785.75
2,BMW,Model B,2006,4.1,98385,Electric,Manual,25760.29
3,Honda,Model B,2015,2.6,88919,Electric,Automatic,25638.00
4,Honda,Model C,2004,3.4,138482,Petrol,Automatic,21021.39
...,...,...,...,...,...,...,...,...
995,Toyota,Model D,2002,1.9,5445,Petrol,Manual,22765.60
996,Honda,Model B,2020,3.1,149112,Diesel,Manual,30392.58
997,Ford,Model C,2008,1.9,195387,Petrol,Automatic,16446.89
998,Toyota,Model A,2003,4.4,246,Petrol,Automatic,27396.16


In [5]:
df.shape
df.info()
df.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Make          1000 non-null   object 
 1   Model         1000 non-null   object 
 2   Year          1000 non-null   int64  
 3   Engine Size   1000 non-null   float64
 4   Mileage       1000 non-null   int64  
 5   Fuel Type     1000 non-null   object 
 6   Transmission  1000 non-null   object 
 7   Price         1000 non-null   float64
dtypes: float64(2), int64(2), object(4)
memory usage: 62.6+ KB


Make            0
Model           0
Year            0
Engine Size     0
Mileage         0
Fuel Type       0
Transmission    0
Price           0
dtype: int64

In [6]:
X = df.drop("Price", axis=1)  
y = df["Price"]

In [7]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [8]:
categorical_cols = ["Make", "Model", "Fuel Type", "Transmission"]
numerical_cols = ["Year", "Engine Size", "Mileage"]

In [9]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

In [10]:
X_train_cat = ohe.fit_transform(X_train[categorical_cols])
X_test_cat = ohe.transform(X_test[categorical_cols])


In [11]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()


In [12]:
X_train_num = scaler.fit_transform(X_train[numerical_cols])
X_test_num = scaler.transform(X_test[numerical_cols])


In [13]:
X_train_final = np.hstack((X_train_num, X_train_cat))
X_test_final = np.hstack((X_test_num, X_test_cat))


In [14]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train_final, y_train)

y_pred_lr = lr.predict(X_test_final)

In [15]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_test, y_pred, model_name):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    print(f"\n--- {model_name} ---")
    print("MAE :", mae)
    print("RMSE:", rmse)
    print("R2  :", r2)

evaluate(y_test, y_pred_lr, "Linear Regression")



--- Linear Regression ---
MAE : 1810.5549626808859
RMSE: 2237.2910362973953
R2  : 0.8170961543618024


In [16]:
from sklearn.linear_model import Ridge

ridge = Ridge(alpha=1.0)
ridge.fit(X_train_final, y_train)

y_pred_ridge = ridge.predict(X_test_final)

evaluate(y_test, y_pred_ridge, "Ridge Regression")



--- Ridge Regression ---
MAE : 1810.185924082315
RMSE: 2236.753067777074
R2  : 0.8171841041900731


In [17]:
from sklearn.linear_model import Lasso

lasso = Lasso(alpha=0.01, max_iter=50000)
lasso.fit(X_train_final, y_train)

y_pred_lasso = lasso.predict(X_test_final)

evaluate(y_test, y_pred_lasso, "Lasso Regression")



--- Lasso Regression ---
MAE : 1810.5438395302315
RMSE: 2237.2788817205287
R2  : 0.8170981416873457


In [18]:
results = pd.DataFrame({
    "Model": ["Linear", "Ridge", "Lasso"],
    "R2 Score": [
        r2_score(y_test, y_pred_lr),
        r2_score(y_test, y_pred_ridge),
        r2_score(y_test, y_pred_lasso)
    ],
    "MAE": [
        mean_absolute_error(y_test, y_pred_lr),
        mean_absolute_error(y_test, y_pred_ridge),
        mean_absolute_error(y_test, y_pred_lasso)
    ]
})

results.sort_values(by="R2 Score", ascending=False)




,Model,R2 Score,MAE
1,Ridge,0.817184,1810.185924
2,Lasso,0.817098,1810.543840
0,Linear,0.817096,1810.554963


In [ ]:
while True:
    make = input("Enter Make (e.g., Honda): ")
    if make.lower() == "exit":
        break

    model = input("Enter Model (e.g., Model B): ")
    if model.lower() == "exit":
        break

    year = input("Enter Year (e.g., 2018): ")
    if year.lower() == "exit":
        break

    engine_size = input("Enter Engine Size (e.g., 2.0): ")
    if engine_size.lower() == "exit":
        break

    mileage = input("Enter Mileage (e.g., 40000): ")
    if mileage.lower() == "exit":
        break

    fuel_type = input("Enter Fuel Type (Petrol/Diesel/Electric): ")
    if fuel_type.lower() == "exit":
        break

    transmission = input("Enter Transmission (Manual/Automatic): ")
    if transmission.lower() == "exit":
        break

    # Convert numeric values
    try:
        year = int(year)
        engine_size = float(engine_size)
        mileage = int(mileage)
    except:
        print("❌ Invalid numeric input. Please enter correct numbers.\n")
        continue

    # Create dataframe for new input
    new_car = pd.DataFrame([{
        "Make": make,
        "Model": model,
        "Year": year,
        "Engine Size": engine_size,
        "Mileage": mileage,
        "Fuel Type": fuel_type,
        "Transmission": transmission
    }])

    # Encode + Scale
    new_cat = ohe.transform(new_car[categorical_cols])
    new_num = scaler.transform(new_car[numerical_cols])
    new_final = np.hstack((new_num, new_cat))

    # Predict
    predicted_price = ridge.predict(new_final)[0]
    print(f"\n✅ Predicted Price: {predicted_price:.2f}\n")

Enter Make (e.g., Honda):  honda
Enter Model (e.g., Model B):  model b
Enter Year (e.g., 2018):  2018
Enter Engine Size (e.g., 2.0):  2.0
Enter Mileage (e.g., 40000):  40000
Enter Fuel Type (Petrol/Diesel/Electric):  petrol
Enter Transmission (Manual/Automatic):  manual



✅ Predicted Price: 30107.65

